Measuring each system's wOBA using RSME. Currently not measuring pitchers or rookies

## Load Packages and Data

In [176]:
import pandas as pd
from pybaseball import batting_stats, playerid_reverse_lookup, playerid_lookup
import numpy as np

In [177]:
pd.set_option('display.max_columns', None)

In [178]:
OOPSY = pd.read_csv("./Projection Systems/OOPSY 2025.csv")
THE_BAT_X = pd.read_csv("./Projection Systems/THE BAT X 2025.csv")
Steamer = pd.read_csv("./Projection Systems/Steamer 2025.csv")
ZiPS = pd.read_csv("./Projection Systems/ZiPS 2025.csv")
ATC = pd.read_csv("./Projection Systems/ATC 2025.csv")
my_system = pd.read_csv("my_system.csv")

#using last year's wOBA as a projection to get baseline for improvement
stats_2024 = batting_stats(2024, 2024, qual=1)
wOBA_2024 = stats_2024[['IDfg', 'wOBA']]
wOBA_2024 = wOBA_2024.rename(columns={"wOBA": "last_year_wOBA"})

In [179]:
actual = batting_stats(2025, 2025, qual=1)
ids = list(actual['IDfg'])
id_data = playerid_reverse_lookup(ids, key_type='fangraphs')
actual = actual[['IDfg', 'Season', 'Name', 'Team', 'Age', 'PA', 'wOBA']]

In [180]:
actual = actual.merge(wOBA_2024, on='IDfg', how='left')

In [181]:
actual = actual.merge(id_data[['key_mlbam', 'key_fangraphs']], left_on='IDfg', right_on='key_fangraphs', how='left')
my_system = my_system.merge(id_data[['key_mlbam', 'key_fangraphs']], left_on='IDfg', right_on='key_fangraphs', how='left')
actual = actual[['key_mlbam', 'Season', 'Name', 'Team', 'Age', 'PA', 'wOBA', 'last_year_wOBA']]

In [182]:
def fill_missing_mlbam_ids(df: pd.DataFrame) -> pd.DataFrame:
    # uses pybaseball's playerid_lookup function to add mlbamids to database using fangraphs ids. 
    # In the case that more than one id is found for a given name, use last year played as 2025. 
    # if still more than one still, use first option then print and personally check
    df = df.copy()
    
    missing_mask = df["key_mlbam"].isna()
    
    for idx, row in df[missing_mask].iterrows():
        full_name = row["Name"]
        
        try:
            first, last = full_name.split(" ", 1)
        except ValueError:
            print(f"Could not split name: {full_name}")
            continue
        
        try:
            lookup = playerid_lookup(last, first, fuzzy=True)

            if len(lookup) > 1:
                
                lookup = lookup[(lookup['key_fangraphs'] == -1) & (lookup['mlb_played_last'] == 2025)]

                if len(lookup) > 1:
                    print(last + " " + first)
                    print(lookup)
            
            if not lookup.empty:
                lookup = lookup[lookup["key_mlbam"].notna()]
                
                if not lookup.empty:
                    df.at[idx, "key_mlbam"] = lookup.iloc[0]["key_mlbam"]
                else:
                    print(f"No valid MLBAM ID found for {full_name}")
            else:
                print(f"No lookup results for {full_name}")
                
        except Exception as e:
            print(f"Error looking up {full_name}: {e}")
    
    return df


In [183]:
actual = fill_missing_mlbam_ids(actual)
my_system = fill_missing_mlbam_ids(my_system)

No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
Pages Pedro
  name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
1     pagés      pedro     686780  pagep001  pagespe02             -1   
3     pagés      pedro     686780  pagep001  pagespe02             -1   

  mlb_played_first mlb_played_last  
1           2024.0          2025.0  
3           2024.0          2025.0  
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identi

## Adjust all systems to assume the same wOBA league average as the 2025 season

In [184]:
actual_league_average = (actual['wOBA'] * actual['PA']).sum() / actual['PA'].sum()

In [185]:
OOPSY_wOBA = OOPSY[['MLBAMID', 'wOBA']]
OOPSY_wOBA = OOPSY_wOBA.rename(columns={"wOBA": 'OOPSY_wOBA'})

THE_BAT_X_wOBA = THE_BAT_X[['MLBAMID', 'wOBA']]
THE_BAT_X_wOBA = THE_BAT_X_wOBA.rename(columns={"wOBA": 'THE_BAT_X_wOBA'})

Steamer_wOBA = Steamer[['MLBAMID', 'wOBA']]
Steamer_wOBA = Steamer_wOBA.rename(columns={"wOBA": 'Steamer_wOBA'})

ZiPS_wOBA = ZiPS[['MLBAMID', 'wOBA']]
ZiPS_wOBA = ZiPS_wOBA.rename(columns={"wOBA": 'ZiPS_wOBA'})

ATC_wOBA = ATC[['MLBAMID', 'wOBA']]
ATC_wOBA = ATC_wOBA.rename(columns={"wOBA": 'ATC_wOBA'})

my_system_wOBA = my_system[['key_mlbam', 'wOBA']]
my_system_wOBA = my_system_wOBA.rename(columns={"wOBA": 'my_system_wOBA'})

In [186]:
OOPSY_wOBA = OOPSY_wOBA.merge(actual[['key_mlbam', 'PA']], left_on='MLBAMID', right_on='key_mlbam')
THE_BAT_X_wOBA = THE_BAT_X_wOBA.merge(actual[['key_mlbam', 'PA']], left_on='MLBAMID', right_on='key_mlbam')
Steamer_wOBA = Steamer_wOBA.merge(actual[['key_mlbam', 'PA']], left_on='MLBAMID', right_on='key_mlbam')
ZiPS_wOBA = ZiPS_wOBA.merge(actual[['key_mlbam', 'PA']], left_on='MLBAMID', right_on='key_mlbam')
ATC_wOBA = ATC_wOBA.merge(actual[['key_mlbam', 'PA']], left_on='MLBAMID', right_on='key_mlbam')
my_system_wOBA = my_system_wOBA.merge(actual[['key_mlbam', 'PA']], on='key_mlbam')


In [187]:
OOPSY_wOBA_league_average = (OOPSY_wOBA['OOPSY_wOBA'] * OOPSY_wOBA['PA']).sum() / OOPSY_wOBA['PA'].sum()
THE_BAT_X_wOBA_league_average = (THE_BAT_X_wOBA['THE_BAT_X_wOBA'] * THE_BAT_X_wOBA['PA']).sum() / THE_BAT_X_wOBA['PA'].sum()
Steamer_wOBA_league_average = (Steamer_wOBA['Steamer_wOBA'] * Steamer_wOBA['PA']).sum() / Steamer_wOBA['PA'].sum()
ZiPS_wOBA_league_average = (ZiPS_wOBA['ZiPS_wOBA'] * ZiPS_wOBA['PA']).sum() / ZiPS_wOBA['PA'].sum()
ATC_wOBA_league_average = (ATC_wOBA['ATC_wOBA'] * ATC_wOBA['PA']).sum() / ATC_wOBA['PA'].sum()
my_system_wOBA_league_average = (my_system_wOBA['my_system_wOBA'] * my_system_wOBA['PA']).sum() / my_system_wOBA['PA'].sum()

In [188]:
OOPSY_wOBA['OOPSY_wOBA'] = OOPSY_wOBA['OOPSY_wOBA'] - OOPSY_wOBA_league_average
OOPSY_wOBA['OOPSY_wOBA'] = OOPSY_wOBA['OOPSY_wOBA'] + actual_league_average

THE_BAT_X_wOBA['THE_BAT_X_wOBA'] = THE_BAT_X_wOBA['THE_BAT_X_wOBA'] - THE_BAT_X_wOBA_league_average
THE_BAT_X_wOBA['THE_BAT_X_wOBA'] = THE_BAT_X_wOBA['THE_BAT_X_wOBA'] + actual_league_average

Steamer_wOBA['Steamer_wOBA'] = Steamer_wOBA['Steamer_wOBA'] - Steamer_wOBA_league_average
Steamer_wOBA['Steamer_wOBA'] = Steamer_wOBA['Steamer_wOBA'] + actual_league_average

ZiPS_wOBA['ZiPS_wOBA'] = ZiPS_wOBA['ZiPS_wOBA'] - ZiPS_wOBA_league_average
ZiPS_wOBA['ZiPS_wOBA'] = ZiPS_wOBA['ZiPS_wOBA'] + actual_league_average

ATC_wOBA['ATC_wOBA'] = ATC_wOBA['ATC_wOBA'] - ATC_wOBA_league_average
ATC_wOBA['ATC_wOBA'] = ATC_wOBA['ATC_wOBA'] + actual_league_average

my_system_wOBA['my_system_wOBA'] = my_system_wOBA['my_system_wOBA'] - my_system_wOBA_league_average
my_system_wOBA['my_system_wOBA'] = my_system_wOBA['my_system_wOBA'] + actual_league_average

In [189]:
OOPSY_wOBA =OOPSY_wOBA.drop(columns=['MLBAMID', 'PA'])
THE_BAT_X_wOBA = THE_BAT_X_wOBA.drop(columns=['MLBAMID', 'PA'])
Steamer_wOBA = Steamer_wOBA.drop(columns=['MLBAMID', 'PA'])
ZiPS_wOBA = ZiPS_wOBA.drop(columns=['MLBAMID', 'PA'])
ATC_wOBA = ATC_wOBA.drop(columns=['MLBAMID', 'PA'])
my_system_wOBA = my_system_wOBA.drop(columns=['PA'])


In [190]:
wOBA_projections = actual.merge(OOPSY_wOBA, on='key_mlbam', how='left')
wOBA_projections = wOBA_projections.merge(THE_BAT_X_wOBA, on='key_mlbam', how='left')
wOBA_projections = wOBA_projections.merge(Steamer_wOBA, on='key_mlbam', how='left')
wOBA_projections = wOBA_projections.merge(ZiPS_wOBA, on='key_mlbam', how='left')
wOBA_projections = wOBA_projections.merge(ATC_wOBA, on='key_mlbam', how='left')
wOBA_projections = wOBA_projections.merge(my_system_wOBA, on='key_mlbam', how='left')

In [191]:
wOBA_projections['OOPSY_wOBA'] = wOBA_projections['OOPSY_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['THE_BAT_X_wOBA'] = wOBA_projections['THE_BAT_X_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['Steamer_wOBA'] = wOBA_projections['Steamer_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['ZiPS_wOBA'] = wOBA_projections['ZiPS_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['ATC_wOBA'] = wOBA_projections['ATC_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['last_year_wOBA'] = wOBA_projections['last_year_wOBA'].fillna(actual_league_average * 0.9)
wOBA_projections['my_system_wOBA'] = wOBA_projections['my_system_wOBA'].fillna(actual_league_average * 0.9)

In [192]:
# OOPSY
weighted_mse_OOPSY = (
    ((wOBA_projections['wOBA'] - wOBA_projections['OOPSY_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_OOPSY = np.sqrt(weighted_mse_OOPSY)

# THE_BAT_X
weighted_mse_THE_BAT_X = (
    ((wOBA_projections['wOBA'] - wOBA_projections['THE_BAT_X_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_THE_BAT_X = np.sqrt(weighted_mse_THE_BAT_X)

# Steamer
weighted_mse_Steamer = (
    ((wOBA_projections['wOBA'] - wOBA_projections['Steamer_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_Steamer = np.sqrt(weighted_mse_Steamer)

# ZiPS
weighted_mse_ZiPS = (
    ((wOBA_projections['wOBA'] - wOBA_projections['ZiPS_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_ZiPS = np.sqrt(weighted_mse_ZiPS)

# ATC
weighted_mse_ATC = (
    ((wOBA_projections['wOBA'] - wOBA_projections['ATC_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_ATC = np.sqrt(weighted_mse_ATC)

#Just using last year's stats as a projection
weighted_mse_last_year = (
    ((wOBA_projections['wOBA'] - wOBA_projections['last_year_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_last_year = np.sqrt(weighted_mse_last_year)

#my system
weighted_mse_my_system = (
    ((wOBA_projections['wOBA'] - wOBA_projections['my_system_wOBA']) ** 2 * wOBA_projections['PA']).sum()
    / wOBA_projections['PA'].sum()
)
weighted_rmse_my_system = np.sqrt(weighted_mse_my_system)

# Print results
print("Weighted RMSE:")
print("OOPSY:", weighted_rmse_OOPSY)
print("THE_BAT_X:", weighted_rmse_THE_BAT_X)
print("Steamer:", weighted_rmse_Steamer)
print("ZiPS:", weighted_rmse_ZiPS)
print("ATC:", weighted_rmse_ATC)
print("Last Year:", weighted_rmse_last_year)
print("my_system:", weighted_rmse_my_system)


Weighted RMSE:
OOPSY: 0.03522268569225316
THE_BAT_X: 0.03546972873056492
Steamer: 0.035498309871067966
ZiPS: 0.035343718127819154
ATC: 0.035417234670440535
Last Year: 0.04791008273520341
my_system: 0.03732341822680167


My system currently does not compare to top systems, improvements can be made on regression to the mean and aging curves